# Day 2 · Session 1B — Fine-tuning Outcome Demo  
## Base T5 vs a T5 Model Fine-tuned for Text-to-SQL

### Why this notebook exists

This notebook demonstrates the **observable effect of fine-tuning** using a publicly available model that has already been trained.

We will compare:

- **Base model:** `t5-base`
- **Fine-tuned model:** `mrm8488/t5-base-finetuned-wikiSQL`

Both use the same T5 architecture, but the second model has been specialized for converting English questions into SQL.

### What participants should observe

Before fine-tuning, the base model is not reliably specialized for Text-to-SQL.

After fine-tuning, the specialized model produces SQL-like output for the same type of input.

> This is the capability shift we want to demonstrate clearly and reliably.

### Important clarification

The public model used here is a **fully fine-tuned checkpoint**, not a LoRA adapter.

The purpose of this notebook is to demonstrate **what specialization looks like**.

LoRA and QLoRA are more efficient ways to reach a similar specialization outcome while training fewer parameters.

## 1. Runtime recommendation

This notebook can run in:

- Google Colab
- VS Code with Jupyter
- JupyterLab

A GPU is helpful but not mandatory because we are only performing inference.

The first run downloads two models, so a stable internet connection is required.

## 2. Install dependencies

If you are using the FDP repository, install the project requirements once:

```bash
python -m pip install -r requirements.txt
```

For Google Colab, uncomment the installation line below.

In [ ]:
# Google Colab only:
# %pip install -q "transformers>=4.44" sentencepiece torch

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("Device:", DEVICE)
print("PyTorch version:", torch.__version__)

## 3. Models used in the comparison

We use the same model family for both sides of the comparison:

```text
T5 Base
   ↓
Fine-tuned on WikiSQL
   ↓
Text-to-SQL capability
```

The specialized checkpoint was fine-tuned on the WikiSQL dataset.

That dataset contains natural-language questions paired with SQL queries.

In [ ]:
BASE_MODEL_NAME = "t5-base"
FINETUNED_MODEL_NAME = "mrm8488/t5-base-finetuned-wikiSQL"

print("Base model:", BASE_MODEL_NAME)
print("Fine-tuned model:", FINETUNED_MODEL_NAME)

## 4. Load the base model

The base model has general text-to-text capabilities.

It has not been specifically specialized in this checkpoint for the Text-to-SQL task.

In [ ]:
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL_NAME)

base_model.to(DEVICE)
base_model.eval()

print("Base model loaded.")

## 5. Load the fine-tuned Text-to-SQL model

This model has the same T5 Base architecture, but its weights have been adapted using WikiSQL examples.

In [ ]:
sql_tokenizer = AutoTokenizer.from_pretrained(FINETUNED_MODEL_NAME)
sql_model = AutoModelForSeq2SeqLM.from_pretrained(FINETUNED_MODEL_NAME)

sql_model.to(DEVICE)
sql_model.eval()

print("Fine-tuned Text-to-SQL model loaded.")

## 6. Create a common generation helper

The Text-to-SQL model expects an instruction prefix:

```text
translate English to SQL:
```

We use the same instruction format for both models so the comparison is fair.

In [ ]:
def generate_text(model, tokenizer, prompt, max_new_tokens=80):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
    ).to(DEVICE)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=4,
            repetition_penalty=1.1,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

## 7. First comparison

We begin with one simple question.

The goal is not to prove perfect SQL correctness.  
The goal is to observe the shift from a general model to a task-specialized model.

In [ ]:
question = "How many employees have a salary greater than 50000?"
prompt = f"translate English to SQL: {question}"

base_output = generate_text(base_model, base_tokenizer, prompt)
finetuned_output = generate_text(sql_model, sql_tokenizer, prompt)

print("QUESTION:")
print(question)

print("\nBASE T5 OUTPUT:")
print(base_output)

print("\nFINE-TUNED T5 OUTPUT:")
print(finetuned_output)

## 8. Test multiple examples

These examples are intentionally simple and similar to the WikiSQL task style.

Expected observation:

- the base model may paraphrase, copy, or produce weak output
- the fine-tuned model should produce SQL-like queries

In [ ]:
TEST_QUESTIONS = [
    "How many students scored more than 80 marks?",
    "What is the average salary of employees in the Engineering department?",
    "Which product has the highest price?",
    "How many courses are offered in semester 6?",
]

results = []

for question in TEST_QUESTIONS:
    prompt = f"translate English to SQL: {question}"

    before = generate_text(
        base_model,
        base_tokenizer,
        prompt,
    )

    after = generate_text(
        sql_model,
        sql_tokenizer,
        prompt,
    )

    results.append({
        "question": question,
        "base_output": before,
        "finetuned_output": after,
    })

    print("=" * 100)
    print("QUESTION:")
    print(question)

    print("\nBEFORE FINE-TUNING — BASE T5:")
    print(before)

    print("\nAFTER FINE-TUNING — TEXT-TO-SQL T5:")
    print(after)

    print()

## 9. Compare the capability, not only the wording

The key question is:

> Did the model begin producing output in the target task language?

For this demo, the target task language is SQL.

We perform a simple check for common SQL keywords.

In [ ]:
SQL_KEYWORDS = [
    "select",
    "from",
    "where",
    "count",
    "avg",
    "max",
    "min",
    "sum",
]

def sql_keyword_score(text):
    lowered = text.lower()
    matched = [keyword for keyword in SQL_KEYWORDS if keyword in lowered]
    return matched

for item in results:
    base_keywords = sql_keyword_score(item["base_output"])
    finetuned_keywords = sql_keyword_score(item["finetuned_output"])

    print("QUESTION:", item["question"])
    print("Base SQL keywords:", base_keywords)
    print("Fine-tuned SQL keywords:", finetuned_keywords)
    print("-" * 100)

## 10. The wow moment

We did not:

- retrain the model in this notebook
- write SQL rules manually
- create a prompt with many examples
- build an SQL parser

We only changed the model checkpoint:

```python
t5-base
```

to:

```python
mrm8488/t5-base-finetuned-wikiSQL
```

That change brought a new specialization.

> **Same architecture. Different learned capability.**

## 11. What exactly changed?

Fine-tuning changed the model weights based on examples of:

```text
English question → SQL query
```

The model learned patterns such as:

- counting rows
- filtering with `WHERE`
- selecting columns
- using aggregate operations
- mapping natural-language phrases to SQL syntax

This does not mean the model understands every database schema.

A production Text-to-SQL system must also receive:

- table names
- column names
- relationships
- constraints
- validation rules

## 12. Fine-tuning vs prompting vs RAG

### Prompt engineering

Use when the base model already has the capability, but needs clearer instructions.

### RAG

Use when the model requires access to changing or private knowledge.

### Fine-tuning

Use when the model must consistently learn a task pattern, behaviour, domain vocabulary, or output capability.

For Text-to-SQL:

- fine-tuning can teach the model the translation task
- schema context is still needed at runtime
- validation must check the generated SQL before execution

## 13. Where LoRA and QLoRA fit

This public checkpoint is a fully fine-tuned model.

LoRA and QLoRA aim to achieve specialization more efficiently:

### Full fine-tuning

- updates all or most model weights
- requires more GPU memory
- saves a complete specialized model

### LoRA

- freezes the base model
- trains small adapter matrices
- stores only the learned adapter

### QLoRA

- keeps the frozen base model in 4-bit form
- trains LoRA adapters
- further reduces memory requirements

The deployment view becomes:

```text
Base Model + LoRA Adapter = Specialized Model
```

## 14. Optional conceptual PEFT loading pattern

The following is the standard pattern for loading a public LoRA adapter.

It is included for explanation only because this notebook uses a full fine-tuned checkpoint.

```python
from peft import PeftModel

base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL_ID)

specialized_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_ID,
)
```

The adapter must be compatible with the exact base model used during training.

## 15. AI Engineering considerations

A fine-tuned model should not be accepted merely because its output looks specialized.

Before production use, validate:

1. **Correctness** — Is the generated SQL valid?
2. **Schema grounding** — Does it use real tables and columns?
3. **Security** — Can it generate destructive queries?
4. **Authorization** — Is the user allowed to access the requested data?
5. **Evaluation** — Does it work on held-out examples?
6. **Monitoring** — Are failures and unsafe queries logged?

For a production system, generated SQL should be validated and executed with least-privilege, read-only access where appropriate.

## 16. Participant activity

Try three new questions:

1. One that requires `COUNT`
2. One that requires `AVG`
3. One that requires `MAX`

Compare the outputs from the base and fine-tuned models.

Discuss:

- Did the fine-tuned model produce SQL-like output?
- Was the SQL logically correct?
- What schema information was missing?
- Would fine-tuning alone be enough for deployment?

# Key takeaways

1. Fine-tuning can create a visible task specialization.
2. The base and fine-tuned models can share the same architecture.
3. Changing the checkpoint can dramatically change model capability.
4. LoRA and QLoRA are efficient ways to produce specialization.
5. Fine-tuning does not remove the need for context, evaluation, validation, and security.
6. In enterprise AI, the real objective is not training—it is reliable task performance.